# MLOps Assignment 2 - DistilBERT Goodreads Genre Classification

### Complete unified solution (Kaggle / Colab / local)

Fine-tune DistilBERT on UCSD Goodreads reviews to classify them into 8 book genres, track the run with Weights & Biases, and publish the model to the Hugging Face Hub.

In [1]:
!pip install -q -U transformers wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 71.2 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()

WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

In [3]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: g25ait2143 (g25ait2143-iitj) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
!pip uninstall -y transformers huggingface_hub
!pip install transformers==4.51.3 huggingface_hub==0.30.2 -q

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: huggingface_hub 1.10.1
Uninstalling huggingface_hub-1.10.1:
  Successfully uninstalled huggingface_hub-1.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.30.2 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.30.2 which is incompatible.


In [5]:
import gzip
import json
import random

import requests
import torch
import wandb
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (DistilBertForSequenceClassification,
                          DistilBertTokenizerFast, Trainer, TrainingArguments)

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 512
SEED = 42
GENRES = ["children", "comics_graphic", "fantasy_paranormal", "history_biography",
          "mystery_thriller_crime", "poetry", "romance", "young_adult"]
NUM_LABELS = len(GENRES)
GENRE_URL_TEMPLATE = ("https://mcauleylab.ucsd.edu/public_datasets/gdrive/"
                      "goodreads/byGenre/goodreads_reviews_{genre}.json.gz")
SAMPLE_SIZE = 1000   # reviews per genre; lower to 200 if GPU hours run low
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

2026-05-27 16:23:17.115074: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779898997.341989      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779898997.405636      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779898997.919564      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779898997.919593      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779898997.919596      23 computation_placer.cc:177] computation placer alr

Device: cuda


In [6]:
def load_reviews(url, head=10000, sample_size=SAMPLE_SIZE):
    """Stream a gzipped genre file and return a random subset of review texts."""
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    reviews = []
    with gzip.open(response.raw, "rt", encoding="utf-8") as handle:
        for count, line in enumerate(handle):
            if count >= head:
                break
            try:
                text = json.loads(line).get("review_text", "").strip()
            except json.JSONDecodeError:
                continue
            if text:
                reviews.append(text)
    return random.sample(reviews, min(sample_size, len(reviews)))


random.seed(SEED)
train_texts, train_labels, test_texts, test_labels = [], [], [], []
train_per_genre = int(SAMPLE_SIZE * 0.8)
for genre in GENRES:
    print("Loading genre:", genre)
    reviews = load_reviews(GENRE_URL_TEMPLATE.format(genre=genre))
    for review in reviews[:train_per_genre]:
        train_texts.append(review)
        train_labels.append(genre)
    for review in reviews[train_per_genre:]:
        test_texts.append(review)
        test_labels.append(genre)
print(f"{len(train_texts)} train / {len(test_texts)} test reviews")

Loading genre: children
Loading genre: comics_graphic
Loading genre: fantasy_paranormal
Loading genre: history_biography
Loading genre: mystery_thriller_crime
Loading genre: poetry
Loading genre: romance
Loading genre: young_adult
6400 train / 1600 test reviews


In [7]:
label2id = {genre: idx for idx, genre in enumerate(GENRES)}
id2label = {idx: genre for genre, idx in label2id.items()}

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
test_enc = tokenizer(test_texts, truncation=True, padding=True, max_length=MAX_LENGTH)


class MyDataset(torch.utils.data.Dataset):
    """Wrap tokenizer encodings + integer labels for the HuggingFace Trainer."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = MyDataset(train_enc, [label2id[label] for label in train_labels])
test_dataset = MyDataset(test_enc, [label2id[label] for label in test_labels])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)
print(model.config.num_labels, "output labels")

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8 output labels


In [9]:
wandb.init(
    project="mlops-assignment2",
    name="distilbert-run-1",
    config={
        "model": MODEL_NAME,
        "epochs": 5,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "max_length": MAX_LENGTH,
        "platform": "Kaggle"
    }
)

wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260527_162343-a6gt5pfq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert-run-1
wandb: ⭐️ View project at https://wandb.ai/g25ait2143-iitj/mlops-assignment2
wandb: 🚀 View run at https://wandb.ai/g25ait2143-iitj/mlops-assignment2/runs/a6gt5pfq
wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="distilbert-run-1"
)

In [11]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.317200,1.281766,0.541875,0.552381
2,1.028200,1.186955,0.576875,0.576910
3,0.701300,1.213153,0.596875,0.596703
4,0.457300,1.308374,0.602500,0.602685
5,0.295200,1.376981,0.600625,0.600664


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=1000, training_loss=0.8542513198852539, metrics={'train_runtime': 939.1965, 'train_samples_per_second': 34.072, 'train_steps_per_second': 1.065, 'total_flos': 4239410331648000.0, 'train_loss': 0.8542513198852539, 'epoch': 5.0})

In [14]:
eval_results = trainer.evaluate()

print(eval_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.1869548559188843, 'eval_accuracy': 0.576875, 'eval_f1': 0.5769095811756596, 'eval_runtime': 14.9444, 'eval_samples_per_second': 107.063, 'eval_steps_per_second': 1.673, 'epoch': 5.0}


In [15]:
preds = trainer.predict(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [16]:
import json
from sklearn.metrics import classification_report

pred_labels = preds.predictions.argmax(-1)

true_labels = [item["labels"].item() for item in test_dataset]

report = classification_report(
    true_labels,
    pred_labels,
    target_names=list(id2label.values()),
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

In [17]:
artifact = wandb.Artifact(
    "eval-report",
    type="evaluation"
)

artifact.add_file("eval_report.json")

wandb.log_artifact(artifact)

<Artifact eval-report>

In [18]:
from huggingface_hub import login

login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [19]:
model.push_to_hub(
    "chaitanyapalagani/distilbert-goodreads-genres"
)

tokenizer.push_to_hub(
    "chaitanyapalagani/distilbert-goodreads-genres"
)

README.md: 0.00B [00:00, ?B/s]

Uploading...:   0%|          | 0.00/263M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/chaitanyapalagani/distilbert-goodreads-genres/commit/4322fb0e806f6ecab35545bf096ad512e544e1fb', commit_message='Upload tokenizer', commit_description='', oid='4322fb0e806f6ecab35545bf096ad512e544e1fb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/chaitanyapalagani/distilbert-goodreads-genres', endpoint='https://huggingface.co', repo_type='model', repo_id='chaitanyapalagani/distilbert-goodreads-genres'), pr_revision=None, pr_num=None)

In [20]:
wandb.run.summary["huggingface_model"] = \
"https://huggingface.co/chaitanyapalagani/distilbert-goodreads-genres"

In [21]:
wandb.finish()

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▅▇██▅
wandb:                 eval/f1 ▁▄▇██▄
wandb:               eval/loss ▄▁▂▅█▁
wandb:            eval/runtime ▁█████
wandb: eval/samples_per_second █▁▁▁▁▁
wandb:   eval/steps_per_second █▁▁▁▁▁
wandb:           test/accuracy ▁
wandb:                 test/f1 ▁
wandb:               test/loss ▁
wandb:            test/runtime ▁
wandb:                      +7 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.57688
wandb:                 eval/f1 0.57691
wandb:               eval/loss 1.18695
wandb:            eval/runtime 14.9444
wandb: eval/samples_per_second 107.063
wandb:   eval/steps_per_second 1.673
wandb:       huggingface_model https://huggingface....
wandb:           test/accuracy 0.57688
wandb:                 test/f1 0.57691
wandb:               test/loss 1.18695
wandb:                     +13 ...


## Submission links

Fill these in after your run, then make every linked resource public:

- **Kaggle Notebook:** `<https://www.kaggle.com/code/nallasairevanth/ml-ops-assig-2>`
- **Hugging Face model:** `<https://huggingface.co/sai5979/distilbert-goodreads-genres>`
- **W&B dashboard:** `<https://wandb.ai/nallasai664-5979/mlops-assignment2/runs/du50y4sr?nw=nwusernallasai664>`
- **GitHub repository:** `<https://github.com/g25ait2141-pixel/mlops-assignment2>`